# CO3133 Assignment 2 (M2 Draft & Controlled Experiment) — Google Colab A100 Runner

**Group:** G-M10 (`Trần Gia Lâm - 2352670`, `Nguyễn Hữu Cầu - 2352129`)  
**Instructor:** Lê Thành Sách  
**Task Track:** Object Detection on **PASCAL VOC 2007** (`VOCtrainval_06-Nov-2007.tar`, 20 classes)  

### What this self-contained notebook does:
1. **Unpacks & verifies** the complete `a2/` pipeline and finalized leak-free split (`4,014` train / `997` validation images, with near-duplicate pair `000863`/`008364` grouped into `train`).
2. **Checks the GPU environment** (target: NVIDIA A100 GPU) and installs any missing lightweight packages (`iterative-stratification`).
3. **Downloads & verifies** the official `VOCtrainval_06-Nov-2007.tar` archive against MD5 `c52e279531787c972589f7e41ab4ae64` and runs a 1-epoch smoke check.
4. **Runs the 200-batch A100 pilot** (`--stage pilot`) to measure step time $t_{\text{step}}$, validation time $t_{\text{val}}$, and peak GPU memory.
5. **Trains and evaluates** the models on the shared validation split using official **VOC2007 11-point interpolated mAP@0.5**:
   - `ssdlite320`: SSDLite320-MobileNetV3-Large lightweight baseline (20 epochs, batch size 8).
   - `fasterrcnn_frozen`: Pretrained Faster R-CNN ResNet-50-FPN (COCO_V1) with frozen ResNet-50 body.
   - `fasterrcnn_finetune`: Pretrained Faster R-CNN ResNet-50-FPN (COCO_V1) with full fine-tuning.
6. **Packages all results** (`metrics.json`, `history.csv`, `per_class_ap.csv`, `comparison.csv`, `curves.png`, `per_class_ap.png`, `examples.png`, `draft_report.md`) into `/content/CO3133_A2_Draft_Outputs.zip` for immediate download.


In [ ]:
# Cell 1: Unpack the verified CO3133_A2_Draft_Starter package and check SHA-256 manifest
import base64, hashlib, json, zipfile
from pathlib import Path

WORK_DIR = Path('/content/CO3133_A2_Draft')
WORK_DIR.mkdir(parents=True, exist_ok=True)
STARTER_ZIP = WORK_DIR / 'CO3133_A2_Draft_Starter.zip'

EMBEDDED_ZIP_B64 = (
    "..."
)
STARTER_ZIP.write_bytes(base64.b64decode(EMBEDDED_ZIP_B64))
with zipfile.ZipFile(STARTER_ZIP, 'r') as zf:
    zf.extractall(WORK_DIR)

manifest = json.loads((WORK_DIR / 'MANIFEST_SHA256.json').read_text(encoding='utf-8'))
for rel_path, expected_hash in manifest.items():
    actual_hash = hashlib.sha256((WORK_DIR / rel_path).read_bytes()).hexdigest()
    assert actual_hash == expected_hash, f'SHA-256 mismatch for {rel_path}'
print(f'Successfully unpacked and verified {len(manifest)} project files in {WORK_DIR}')


Successfully unpacked and verified 14 project files in /content/CO3133_A2_Draft


In [ ]:
# Cell 2: Enter workspace, install lightweight dependencies, and verify GPU runtime
%cd /content/CO3133_A2_Draft
!pip install -q iterative-stratification Markdown

import torch, torchvision
print('PyTorch version:', torch.__version__)
print('Torchvision version:', torchvision.__version__)
assert torch.cuda.is_available(), 'CUDA is unavailable! Please select a GPU runtime (Runtime -> Change runtime type -> A100 / GPU).'
print('Assigned GPU:', torch.cuda.get_device_name(0))
print(f'Total GPU VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB')


/content/CO3133_A2_Draft
PyTorch version: 2.11.0+cu130
Torchvision version: 0.26.0+cu130
Assigned GPU: NVIDIA A100-SXM4-40GB
Total GPU VRAM: 39.49 GB


In [ ]:
# Cell 3: Run VOC2007 Evaluator unit tests, download & verify VOCtrainval_06-Nov-2007.tar, and export batch preview
!python run_a2.py --stage eda --device cuda --out outputs/a2_m2_a100


{
  "group": "G-M10",
  "members": [
    "Trần Gia Lâm (2352670)",
    "Nguyễn Hữu Cầu (2352129)"
  ],
  "seed": 42,
  "dataset": "PASCAL VOC 2007",
  "archive_name": "VOCtrainval_06-Nov-2007.tar",
  "archive_url": "https://thor.robots.ox.ac.uk/pascal/VOC/voc2007/VOCtrainval_06-Nov-2007.tar",
  "archive_md5": "c52e279531787c972589f7e41ab4ae64",
  "num_classes": 21,
  "iou_threshold": 0.5,
  "score_threshold": 0.05,
  "nms_threshold": 0.5,
  "max_detections_per_image": 100,
  "pilot_batches": 200,
  "num_workers": 2,
  "cpu_threads": 4,
  "use_amp": true,
  "device": "cuda",
  "models": {
    "ssdlite320": {
      "architecture": "ssdlite320_mobilenet_v3_large",
      "init_weights": "MobileNet_V3_Large_Weights.IMAGENET1K_V1 backbone + fresh 21-class heads",
      "freeze_backbone": false,
      "min_size": 320,
      "max_size": 320,
      "batch_size": 8,
      "epochs": 20,
      "optimizer": "SGD",
      "learning_rate": 0.01,
      "momentum": 0.9,
      "weight_decay": 0.0005,
   

In [ ]:
# Cell 4: Run the 200-batch + 1-validation-pass pilot per configuration (Proposal Section 3.1)
!python run_a2.py --stage pilot --device cuda --out outputs/a2_m2_a100


{
  "group": "G-M10",
  "members": [
    "Trần Gia Lâm (2352670)",
    "Nguyễn Hữu Cầu (2352129)"
  ],
  "seed": 42,
  "dataset": "PASCAL VOC 2007",
  "archive_name": "VOCtrainval_06-Nov-2007.tar",
  "archive_url": "https://thor.robots.ox.ac.uk/pascal/VOC/voc2007/VOCtrainval_06-Nov-2007.tar",
  "archive_md5": "c52e279531787c972589f7e41ab4ae64",
  "num_classes": 21,
  "iou_threshold": 0.5,
  "score_threshold": 0.05,
  "nms_threshold": 0.5,
  "max_detections_per_image": 100,
  "pilot_batches": 200,
  "num_workers": 2,
  "cpu_threads": 4,
  "use_amp": true,
  "device": "cuda",
  "models": {
    "ssdlite320": {
      "architecture": "ssdlite320_mobilenet_v3_large",
      "init_weights": "MobileNet_V3_Large_Weights.IMAGENET1K_V1 backbone + fresh 21-class heads",
      "freeze_backbone": false,
      "min_size": 320,
      "max_size": 320,
      "batch_size": 8,
      "epochs": 20,
      "optimizer": "SGD",
      "learning_rate": 0.01,
      "momentum": 0.9,
      "weight_decay": 0.0005,
   

In [12]:
# Cell 5: Train SSDLite320 baseline and Faster R-CNN (frozen backbone vs. full fine-tuning)
!python run_a2.py --stage train --device cuda --out outputs/a2_m2_a100


{
  "group": "G-M10",
  "members": [
    "Trần Gia Lâm (2352670)",
    "Nguyễn Hữu Cầu (2352129)"
  ],
  "seed": 42,
  "dataset": "PASCAL VOC 2007",
  "archive_name": "VOCtrainval_06-Nov-2007.tar",
  "archive_url": "https://thor.robots.ox.ac.uk/pascal/VOC/voc2007/VOCtrainval_06-Nov-2007.tar",
  "archive_md5": "c52e279531787c972589f7e41ab4ae64",
  "num_classes": 21,
  "iou_threshold": 0.5,
  "score_threshold": 0.05,
  "nms_threshold": 0.5,
  "max_detections_per_image": 100,
  "pilot_batches": 200,
  "num_workers": 2,
  "cpu_threads": 4,
  "use_amp": true,
  "device": "cuda",
  "models": {
    "ssdlite320": {
      "architecture": "ssdlite320_mobilenet_v3_large",
      "init_weights": "MobileNet_V3_Large_Weights.IMAGENET1K_V1 backbone + fresh 21-class heads",
      "freeze_backbone": false,
      "min_size": 320,
      "max_size": 320,
      "batch_size": 8,
      "epochs": 20,
      "optimizer": "SGD",
      "learning_rate": 0.01,
      "momentum": 0.9,
      "weight_decay": 0.0005,
   

In [13]:
# Cell 6: Reload saved best.pt checkpoints and verify validation mAP@0.5 reproducibility
!python run_a2.py --stage evaluate --device cuda --out outputs/a2_m2_a100


{
  "group": "G-M10",
  "members": [
    "Trần Gia Lâm (2352670)",
    "Nguyễn Hữu Cầu (2352129)"
  ],
  "seed": 42,
  "dataset": "PASCAL VOC 2007",
  "archive_name": "VOCtrainval_06-Nov-2007.tar",
  "archive_url": "https://thor.robots.ox.ac.uk/pascal/VOC/voc2007/VOCtrainval_06-Nov-2007.tar",
  "archive_md5": "c52e279531787c972589f7e41ab4ae64",
  "num_classes": 21,
  "iou_threshold": 0.5,
  "score_threshold": 0.05,
  "nms_threshold": 0.5,
  "max_detections_per_image": 100,
  "pilot_batches": 200,
  "num_workers": 2,
  "cpu_threads": 4,
  "use_amp": true,
  "device": "cuda",
  "models": {
    "ssdlite320": {
      "architecture": "ssdlite320_mobilenet_v3_large",
      "init_weights": "MobileNet_V3_Large_Weights.IMAGENET1K_V1 backbone + fresh 21-class heads",
      "freeze_backbone": false,
      "min_size": 320,
      "max_size": 320,
      "batch_size": 8,
      "epochs": 20,
      "optimizer": "SGD",
      "learning_rate": 0.01,
      "momentum": 0.9,
      "weight_decay": 0.0005,
   

In [14]:
# Cell 7: Inspect summary metrics, per-class AP, and error taxonomy
import json
from pathlib import Path

out_dir = Path('outputs/a2_m2_a100')
print((out_dir / 'comparison.md').read_text(encoding='utf-8'))

for model_name in ['ssdlite320', 'fasterrcnn_frozen', 'fasterrcnn_finetune']:
    m_path = out_dir / model_name / 'metrics.json'
    if m_path.exists():
        m = json.loads(m_path.read_text(encoding='utf-8'))
        print(f'=== {model_name.upper()} ===')
        print(f"  Best epoch       : {m['best_epoch']} / {m['epochs_completed']}")
        print(f"  Val mAP@0.5      : {m['map50']*100:.2f}% (Small: {m['map50_small']*100:.2f}%, Med: {m['map50_medium']*100:.2f}%, Large: {m['map50_large']*100:.2f}%)")
        print(f"  Params (tr/tot)  : {m['trainable_parameters']:,} / {m['total_parameters']:,}")
        print(f"  Train / Val time : {m['train_seconds']:.1f}s / {m['validation_seconds']:.1f}s (Peak VRAM: {m['peak_gpu_memory_mb']:.0f} MB)")
        print(f"  Inference latency: {m['inference']['mean_latency_ms']:.2f} ms/img ({m['inference']['fps']:.1f} FPS)")
        print(f"  Error taxonomy   : {m['error_taxonomy']}")


| Model | Backbone mode | Val mAP@0.5 | Small AP | Med AP | Large AP | Trainable / Total Params | Best Epoch | Train (s) | Latency (ms/img) | Peak VRAM (MB) |
|---|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| `ssdlite320` | ImageNet backbone | **51.53%** | 2.29% | 34.36% | 61.96% | 4,026,740 / 4,026,740 | 18/20 | 832.3 | 16.62 | 463 |
| `fasterrcnn_frozen` | Frozen ResNet-50 | **83.63%** | 50.08% | 74.51% | 82.24% | 17,941,624 / 41,396,536 | 12/12 | 1320.3 | 29.28 | 1266 |
| `fasterrcnn_finetune` | Full fine-tune | **75.07%** | 42.14% | 62.14% | 73.95% | 41,396,536 / 41,396,536 | 10/12 | 1807.6 | 22.03 | 3613 |

=== SSDLITE320 ===
  Best epoch       : 18 / 20
  Val mAP@0.5      : 51.53% (Small: 2.29%, Med: 34.36%, Large: 61.96%)
  Params (tr/tot)  : 4,026,740 / 4,026,740
  Train / Val time : 832.3s / 238.7s (Peak VRAM: 463 MB)
  Inference latency: 16.62 ms/img (60.2 FPS)
  Error taxonomy   : {'score_threshold_used': 0.3, 'total_non_difficult_gt': 2592, 'correct_detections': 1391

In [15]:
# Cell 8: Package all report outputs (JSONs, CSVs, PNG figures, markdown report) into a lightweight ZIP and download
import zipfile
from pathlib import Path

out_root = Path('/content/CO3133_A2_Draft/outputs/a2_m2_a100')
report_zip = Path('/content/CO3133_A2_Draft_Outputs.zip')

with zipfile.ZipFile(report_zip, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for file_path in sorted(out_root.rglob('*')):
        if file_path.is_file() and file_path.suffix != '.pt':
            arc = Path('outputs/a2_m2_a100') / file_path.relative_to(out_root)
            zf.write(file_path, arcname=arc.as_posix())

print(f'Created {report_zip} ({report_zip.stat().st_size / 1024:.1f} KB)')
try:
    from google.colab import files
    files.download(str(report_zip))
except Exception as e:
    print('Download via Files panel:', report_zip, e)


Created /content/CO3133_A2_Draft_Outputs.zip (13092.5 KB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>